# FEEDIT 상품명 EDA Notebook

CSV에서 플랫폼 상품명 데이터를 읽어 **정규화 전에 상품명 구조와 어휘 분포를 파악**하기 위한 분석용 노트북입니다.

주요 분석:
1. 기본 데이터 품질
2. 상품명 길이 / 특수문자 / 숫자 / 괄호 구조
3. `[]`, `()`, BODY 분리
4. 토큰 빈도 / 문서 빈도
5. Bigram / Trigram
6. 플랫폼별 비교
7. 카테고리별 특징어
8. TF-IDF
9. Co-occurrence
10. 사전 Coverage / Unknown Candidate
11. 정규화 규칙 후보 추출

## 0. 환경 설정

In [ ]:
# 필요 시 설치
# %pip install pandas numpy matplotlib scikit-learn kiwipiepy openpyxl

import re
import math
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

## 1. CSV 로드

아래 4개 컬럼명만 실제 CSV에 맞게 수정하면 됩니다.

- `NAME_COL`: 플랫폼 상품명
- `PLATFORM_COL`: 플랫폼명
- `CATEGORY_COL`: 플랫폼 카테고리
- `ID_COL`: 플랫폼 상품 ID

In [ ]:
CSV_PATH = "product_source.csv"

NAME_COL = "platform_product_name"
PLATFORM_COL = "platform"
CATEGORY_COL = "platform_category"
ID_COL = "platform_product_id"

df = pd.read_csv(CSV_PATH)

print("shape:", df.shape)
display(df.head())
display(df.dtypes)

## 2. 최소 컬럼 정리

In [ ]:
required = [NAME_COL]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

work = df.copy()

work["raw_name"] = work[NAME_COL].fillna("").astype(str)

if PLATFORM_COL in work.columns:
    work["platform_value"] = work[PLATFORM_COL].fillna("UNKNOWN").astype(str)
else:
    work["platform_value"] = "UNKNOWN"

if CATEGORY_COL in work.columns:
    work["category_value"] = work[CATEGORY_COL].fillna("UNKNOWN").astype(str)
else:
    work["category_value"] = "UNKNOWN"

if ID_COL in work.columns:
    work["product_id_value"] = work[ID_COL]
else:
    work["product_id_value"] = np.arange(len(work))

display(work[["product_id_value", "platform_value", "category_value", "raw_name"]].head())

## 3. 기본 데이터 품질

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "rows",
        "empty_name",
        "duplicate_name",
        "unique_name",
        "unique_platform",
        "unique_category",
    ],
    "value": [
        len(work),
        (work["raw_name"].str.strip() == "").sum(),
        work["raw_name"].duplicated().sum(),
        work["raw_name"].nunique(),
        work["platform_value"].nunique(),
        work["category_value"].nunique(),
    ]
})

display(summary)

print("\n플랫폼별 건수")
display(work["platform_value"].value_counts().rename_axis("platform").to_frame("count").head(30))

print("\n카테고리별 건수")
display(work["category_value"].value_counts().rename_axis("category").to_frame("count").head(50))

## 4. 상품명 문자열 프로파일링

In [ ]:
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\u2600-\u26FF"
    "\u2700-\u27BF"
    "]+"
)

def count_emoji(text):
    return len(EMOJI_RE.findall(text))

def count_korean(text):
    return len(re.findall(r"[가-힣]", text))

def count_english(text):
    return len(re.findall(r"[A-Za-z]", text))

def count_digits(text):
    return len(re.findall(r"\d", text))

work["name_length"] = work["raw_name"].str.len()
work["space_token_count"] = work["raw_name"].str.split().str.len()
work["square_bracket_count"] = work["raw_name"].str.count(r"\[")
work["paren_count"] = work["raw_name"].str.count(r"\(")
work["slash_count"] = work["raw_name"].str.count("/")
work["emoji_count"] = work["raw_name"].map(count_emoji)
work["digit_count"] = work["raw_name"].map(count_digits)
work["english_count"] = work["raw_name"].map(count_english)
work["korean_count"] = work["raw_name"].map(count_korean)

profile_cols = [
    "name_length", "space_token_count", "square_bracket_count",
    "paren_count", "slash_count", "emoji_count",
    "digit_count", "english_count", "korean_count"
]

display(work[profile_cols].describe().T)

## 5. 상품명 길이 분포

In [ ]:
plt.figure(figsize=(10, 5))
work["name_length"].clip(upper=work["name_length"].quantile(0.99)).hist(bins=50)
plt.title("Product Name Length Distribution")
plt.xlabel("Length")
plt.ylabel("Count")
plt.show()

display(
    work.nlargest(30, "name_length")[
        ["platform_value", "category_value", "name_length", "raw_name"]
    ]
)

## 6. 특수 구조 포함률

In [ ]:
pattern_rate = pd.DataFrame({
    "pattern": ["[]", "()", "/", "emoji", "digit", "english"],
    "count": [
        (work["square_bracket_count"] > 0).sum(),
        (work["paren_count"] > 0).sum(),
        (work["slash_count"] > 0).sum(),
        (work["emoji_count"] > 0).sum(),
        (work["digit_count"] > 0).sum(),
        (work["english_count"] > 0).sum(),
    ]
})
pattern_rate["rate"] = pattern_rate["count"] / len(work)

display(pattern_rate.sort_values("rate", ascending=False))

## 7. `[ ]`, `( )`, BODY 구조 분리

In [ ]:
def extract_square_blocks(text):
    return re.findall(r"\[([^\]]+)\]", text)

def extract_paren_blocks(text):
    return re.findall(r"\(([^)]+)\)", text)

def extract_body(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

work["square_blocks"] = work["raw_name"].map(extract_square_blocks)
work["paren_blocks"] = work["raw_name"].map(extract_paren_blocks)
work["body"] = work["raw_name"].map(extract_body)

display(
    work[["raw_name", "square_blocks", "body", "paren_blocks"]].head(30)
)

## 8. 영역별 텍스트 데이터셋 만들기

In [ ]:
square_texts = [
    block
    for blocks in work["square_blocks"]
    for block in blocks
]

paren_texts = [
    block
    for blocks in work["paren_blocks"]
    for block in blocks
]

body_texts = work["body"].tolist()

print("square blocks:", len(square_texts))
print("paren blocks:", len(paren_texts))
print("body rows:", len(body_texts))

## 9. 기본 토크나이저

In [ ]:
TOKEN_RE = re.compile(r"[가-힣A-Za-z0-9]+(?:[-+][가-힣A-Za-z0-9]+)*")

def basic_tokenize(text):
    text = unicodedata.normalize("NFKC", str(text))
    return [t.lower() for t in TOKEN_RE.findall(text)]

work["tokens"] = work["raw_name"].map(basic_tokenize)
work["body_tokens"] = work["body"].map(basic_tokenize)

display(work[["raw_name", "tokens"]].head(20))

## 10. Unigram 빈도

In [ ]:
def token_frequency(token_lists):
    return Counter(t for tokens in token_lists for t in tokens)

token_tf = token_frequency(work["tokens"])
body_tf = token_frequency(work["body_tokens"])

tf_df = pd.DataFrame(token_tf.most_common(), columns=["token", "tf"])
body_tf_df = pd.DataFrame(body_tf.most_common(), columns=["token", "body_tf"])

display(tf_df.head(100))

## 11. Document Frequency

In [ ]:
token_df = Counter()

for tokens in work["tokens"]:
    token_df.update(set(tokens))

df_df = pd.DataFrame(token_df.items(), columns=["token", "df"])
df_df["df_rate"] = df_df["df"] / len(work)

token_stats = (
    tf_df
    .merge(df_df, on="token", how="left")
    .merge(body_tf_df, on="token", how="left")
    .fillna(0)
    .sort_values(["df", "tf"], ascending=False)
)

display(token_stats.head(150))

## 12. `[]` / `()` / BODY 영역별 주요 단어

In [ ]:
square_tokens = [basic_tokenize(x) for x in square_texts]
paren_tokens = [basic_tokenize(x) for x in paren_texts]

square_freq = pd.DataFrame(
    token_frequency(square_tokens).most_common(100),
    columns=["token", "square_tf"]
)

paren_freq = pd.DataFrame(
    token_frequency(paren_tokens).most_common(100),
    columns=["token", "paren_tf"]
)

body_freq = pd.DataFrame(
    token_frequency(work["body_tokens"]).most_common(100),
    columns=["token", "body_tf"]
)

print("[] TOP")
display(square_freq)

print("() TOP")
display(paren_freq)

print("BODY TOP")
display(body_freq)

## 13. Bigram / Trigram

In [ ]:
def make_ngrams(tokens, n=2):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

bigram_counter = Counter()
trigram_counter = Counter()

for tokens in work["body_tokens"]:
    bigram_counter.update(make_ngrams(tokens, 2))
    trigram_counter.update(make_ngrams(tokens, 3))

bigram_df = pd.DataFrame(
    bigram_counter.most_common(200),
    columns=["bigram", "count"]
)

trigram_df = pd.DataFrame(
    trigram_counter.most_common(200),
    columns=["trigram", "count"]
)

display(bigram_df.head(100))
display(trigram_df.head(100))

## 14. 플랫폼별 문자열 구조 비교

In [ ]:
platform_profile = (
    work.groupby("platform_value")[profile_cols]
    .mean()
    .round(2)
)

display(platform_profile)

platform_pattern = (
    work.assign(
        has_square=work["square_bracket_count"] > 0,
        has_paren=work["paren_count"] > 0,
        has_slash=work["slash_count"] > 0,
        has_emoji=work["emoji_count"] > 0,
        has_digit=work["digit_count"] > 0,
    )
    .groupby("platform_value")[
        ["has_square", "has_paren", "has_slash", "has_emoji", "has_digit"]
    ]
    .mean()
    .round(4)
)

display(platform_pattern)

## 15. 플랫폼별 TOP Token

In [ ]:
def top_tokens_by_group(frame, group_col, token_col="tokens", top_n=30):
    rows = []

    for group, sub in frame.groupby(group_col):
        counter = Counter(
            token
            for tokens in sub[token_col]
            for token in tokens
        )

        for token, count in counter.most_common(top_n):
            rows.append({
                group_col: group,
                "token": token,
                "count": count
            })

    return pd.DataFrame(rows)

platform_tokens = top_tokens_by_group(
    work,
    "platform_value",
    "tokens",
    top_n=50
)

display(platform_tokens)

## 16. 카테고리별 TOP Token

In [ ]:
category_tokens = top_tokens_by_group(
    work,
    "category_value",
    "body_tokens",
    top_n=30
)

display(category_tokens.head(300))

## 17. 카테고리별 TF-IDF 특징어

In [ ]:
category_docs = (
    work.groupby("category_value")["body"]
    .apply(lambda x: " ".join(x.astype(str)))
)

vectorizer = TfidfVectorizer(
    tokenizer=basic_tokenize,
    token_pattern=None,
    lowercase=False,
    min_df=1,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(category_docs.values)
features = np.array(vectorizer.get_feature_names_out())

tfidf_rows = []

for idx, category in enumerate(category_docs.index):
    row = X[idx].toarray().ravel()
    top_idx = row.argsort()[::-1][:30]

    for rank, j in enumerate(top_idx, start=1):
        if row[j] <= 0:
            continue
        tfidf_rows.append({
            "category": category,
            "rank": rank,
            "term": features[j],
            "tfidf": row[j],
        })

category_tfidf = pd.DataFrame(tfidf_rows)

display(category_tfidf.head(300))

## 18. Token 위치 분석

In [ ]:
position_rows = []

for row_idx, tokens in work["tokens"].items():
    n = len(tokens)

    if n <= 1:
        continue

    for i, token in enumerate(tokens):
        position_rows.append({
            "token": token,
            "position": i / (n - 1)
        })

position_df = pd.DataFrame(position_rows)

position_stats = (
    position_df.groupby("token")
    .agg(
        count=("position", "size"),
        mean_position=("position", "mean"),
        median_position=("position", "median"),
    )
    .query("count >= 20")
    .sort_values("count", ascending=False)
)

display(position_stats.head(200))

## 19. 앞 / 중간 / 뒤에 자주 나오는 단어

In [ ]:
front_tokens = position_stats.query("mean_position <= 0.25").sort_values("count", ascending=False)
middle_tokens = position_stats.query("0.35 <= mean_position <= 0.65").sort_values("count", ascending=False)
back_tokens = position_stats.query("mean_position >= 0.75").sort_values("count", ascending=False)

print("FRONT")
display(front_tokens.head(100))

print("MIDDLE")
display(middle_tokens.head(100))

print("BACK")
display(back_tokens.head(100))

## 20. Co-occurrence

In [ ]:
MIN_TOKEN_DF = 10

valid_tokens = {
    token for token, freq in token_df.items()
    if freq >= MIN_TOKEN_DF
}

pair_counter = Counter()

for tokens in work["body_tokens"]:
    unique_tokens = sorted(set(tokens) & valid_tokens)

    for i in range(len(unique_tokens)):
        for j in range(i + 1, len(unique_tokens)):
            pair_counter[(unique_tokens[i], unique_tokens[j])] += 1

cooc_df = pd.DataFrame(
    [
        {"token_a": a, "token_b": b, "count": count}
        for (a, b), count in pair_counter.most_common(1000)
    ]
)

display(cooc_df.head(200))

## 21. 특정 단어 연관어 조회

In [ ]:
def related_tokens(keyword, top_n=30):
    rows = []

    for (a, b), count in pair_counter.items():
        if a == keyword:
            rows.append((b, count))
        elif b == keyword:
            rows.append((a, count))

    return pd.DataFrame(
        sorted(rows, key=lambda x: x[1], reverse=True)[:top_n],
        columns=["related_token", "cooccurrence"]
    )

# 예시
# display(related_tokens("가디건"))
# display(related_tokens("부츠컷"))

## 22. FEEDIT Dictionary Coverage

사전 CSV를 준비했다면 아래처럼 연결합니다.

예시 포맷:

| term | term_type |
|---|---|
| 가디건 | ITEM |
| 부츠컷 | FIT |
| 스웨이드 | MATERIAL |
| 발레코어 | STYLE |

In [ ]:
DICT_PATH = None
# DICT_PATH = "feedit_term_dict.csv"

TERM_COL = "term"
TERM_TYPE_COL = "term_type"

if DICT_PATH:
    term_dict = pd.read_csv(DICT_PATH)
    term_dict[TERM_COL] = term_dict[TERM_COL].astype(str).str.lower().str.strip()

    known_terms = set(term_dict[TERM_COL])

    unique_tokens = set(token_stats["token"])

    known = unique_tokens & known_terms
    unknown = unique_tokens - known_terms

    coverage = pd.DataFrame({
        "metric": [
            "unique_token_count",
            "known_token_count",
            "unknown_token_count",
            "unique_token_coverage"
        ],
        "value": [
            len(unique_tokens),
            len(known),
            len(unknown),
            len(known) / max(len(unique_tokens), 1)
        ]
    })

    display(coverage)

    unknown_stats = (
        token_stats[token_stats["token"].isin(unknown)]
        .sort_values(["df", "tf"], ascending=False)
    )

    display(unknown_stats.head(300))
else:
    print("DICT_PATH를 지정하면 Dictionary Coverage 분석이 실행됩니다.")

## 23. 신규 Term Candidate 후보

In [ ]:
if DICT_PATH:
    candidate_terms = (
        unknown_stats
        .query("df >= 3")
        .copy()
    )

    candidate_terms["candidate_score"] = (
        np.log1p(candidate_terms["df"]) *
        np.log1p(candidate_terms["tf"])
    )

    candidate_terms = candidate_terms.sort_values(
        "candidate_score",
        ascending=False
    )

    display(candidate_terms.head(300))

## 24. 마케팅 / Noise 후보 탐색

In [ ]:
# [] 영역에는 광고성 단어가 자주 나타날 가능성이 있으므로,
# square 영역 빈도와 BODY 영역 빈도의 비율을 이용해 후보를 볼 수 있습니다.

square_all = Counter(t for tokens in square_tokens for t in tokens)
body_all = Counter(t for tokens in work["body_tokens"] for t in tokens)

noise_rows = []

vocab = set(square_all) | set(body_all)

for token in vocab:
    square_count = square_all[token]
    body_count = body_all[token]

    total = square_count + body_count

    if total < 5:
        continue

    noise_rows.append({
        "token": token,
        "square_tf": square_count,
        "body_tf": body_count,
        "square_ratio": square_count / total,
        "total_tf": total
    })

noise_candidates = (
    pd.DataFrame(noise_rows)
    .sort_values(["square_ratio", "total_tf"], ascending=[False, False])
)

display(noise_candidates.head(200))

## 25. 숫자 / 할인 / 판매량 패턴 탐색

In [ ]:
regex_patterns = {
    "percent": r"\b\d{1,3}\s*%",
    "color_count": r"\b\d+\s*colors?\b|\b\d+\s*컬러\b",
    "size_range": r"\b(?:xs|s|m|l|xl|xxl)\s*[-~]\s*(?:xs|s|m|l|xl|xxl)\b",
    "sales_count": r"\d+\s*(?:천|만)?\s*장\s*(?:돌파|판매)?",
}

pattern_rows = []

for name, pattern in regex_patterns.items():
    matches = work["raw_name"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

    pattern_rows.append({
        "pattern": name,
        "count": matches.sum(),
        "rate": matches.mean()
    })

display(pd.DataFrame(pattern_rows))

## 26. Emoji 빈도

In [ ]:
emoji_counter = Counter()

for text in work["raw_name"]:
    emoji_counter.update(EMOJI_RE.findall(text))

emoji_df = pd.DataFrame(
    emoji_counter.most_common(),
    columns=["emoji", "count"]
)

display(emoji_df.head(100))

## 27. 분석 결과 CSV 저장

In [ ]:
OUTPUT_DIR = Path("eda_output")
OUTPUT_DIR.mkdir(exist_ok=True)

token_stats.to_csv(
    OUTPUT_DIR / "token_stats.csv",
    index=False,
    encoding="utf-8-sig"
)

bigram_df.to_csv(
    OUTPUT_DIR / "bigram_top.csv",
    index=False,
    encoding="utf-8-sig"
)

trigram_df.to_csv(
    OUTPUT_DIR / "trigram_top.csv",
    index=False,
    encoding="utf-8-sig"
)

platform_tokens.to_csv(
    OUTPUT_DIR / "platform_token_top.csv",
    index=False,
    encoding="utf-8-sig"
)

category_tokens.to_csv(
    OUTPUT_DIR / "category_token_top.csv",
    index=False,
    encoding="utf-8-sig"
)

category_tfidf.to_csv(
    OUTPUT_DIR / "category_tfidf.csv",
    index=False,
    encoding="utf-8-sig"
)

position_stats.reset_index().to_csv(
    OUTPUT_DIR / "token_position_stats.csv",
    index=False,
    encoding="utf-8-sig"
)

cooc_df.to_csv(
    OUTPUT_DIR / "cooccurrence_top.csv",
    index=False,
    encoding="utf-8-sig"
)

noise_candidates.to_csv(
    OUTPUT_DIR / "noise_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

if DICT_PATH:
    candidate_terms.to_csv(
        OUTPUT_DIR / "unknown_term_candidates.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("저장 완료:", OUTPUT_DIR.resolve())

# 다음 분석 확장 후보

이 노트북 결과를 본 뒤 다음 순서로 확장하면 됩니다.

- Kiwi 형태소 분석 결과와 기본 tokenizer 비교
- FEEDIT Dictionary longest-match tokenizer
- 카테고리 × 속성 연관 분석
- PMI 기반 Co-occurrence
- 플랫폼별 상품명 문법 비교
- 광고성 Prefix / Suffix 자동 탐지
- Unknown Term Candidate 자동 승격 파이프라인 연결
- 정규화 전/후 정보 손실률 비교